In [13]:
import numpy as np
import matplotlib.pyplot as plt
import json
import os
from mpl_toolkits.mplot3d import Axes3D, proj3d
from PIL import Image
import random
import cv2  

dataset_size = 5000                   # Количество изображений за прогон
start_index = 0                       # С какой позиции продолжать генерацию
image_size = (256, 256)
output_dir = 'cube_dataset'
background_dir = 'fons'

base_noise_level = 0.02              # Базовый уровень "ручного" искажения вершин куба
base_line_width = 0.55               # Базовая толщина линий
base_texture_jitter = 0.005          # Базовая сила дрожания линий
segments_per_edge_range = (30, 40)   # Диапазон количества отрезков на ребре
visibility_threshold = 2.0           # Порог перекрытия в пикселях (для рёбер куба)
       
image_noise_density = 0.5            # Интенсивность шума: 0 – без шума, большее значение – больше шумовых элементов
noise_color_mode = 'bw'              # Режим цвета шума: 'bw' – оттенки серого, 'color' – случайные тёмные цвета

os.makedirs(output_dir, exist_ok=True)
os.makedirs(f'{output_dir}/images', exist_ok=True)
os.makedirs(f'{output_dir}/images/points', exist_ok=True)
os.makedirs(f'{output_dir}/images/cubes', exist_ok=True)
os.makedirs(f'{output_dir}/annotations', exist_ok=True)

if os.path.exists(background_dir):
    background_images = [os.path.join(background_dir, f)
                         for f in os.listdir(background_dir)
                         if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
else:
    background_images = []

def create_noisy_cube():
    """Создаёт координаты вершин куба с добавлением случайного шума."""
    base_vertices = np.array([
        [-1, -1, -1], [1, -1, -1], [1, 1, -1], [-1, 1, -1],
        [-1, -1,  1], [1, -1,  1], [1, 1,  1], [-1, 1,  1]
    ])
    scale = np.random.uniform(0.5, 1.5)
    scaled_vertices = base_vertices * scale
    noise = np.random.uniform(-base_noise_level, base_noise_level, scaled_vertices.shape)
    return scaled_vertices + noise

def random_rotation_matrix():
    """Создаёт случайную матрицу поворота."""
    alpha, beta, gamma = np.random.uniform(0, 2 * np.pi, 3)
    Rx = np.array([
        [1, 0, 0],
        [0, np.cos(alpha), -np.sin(alpha)],
        [0, np.sin(alpha), np.cos(alpha)]
    ])
    Ry = np.array([
        [np.cos(beta), 0, np.sin(beta)],
        [0, 1, 0],
        [-np.sin(beta), 0, np.cos(beta)]
    ])
    Rz = np.array([
        [np.cos(gamma), -np.sin(gamma), 0],
        [np.sin(gamma),  np.cos(gamma), 0],
        [0, 0, 1]
    ])
    return Rz @ Ry @ Rx

def get_random_background_crop():
    """Возвращает случайную обрезку фонового изображения под заданный размер."""
    if background_images:
        bg_path = random.choice(background_images)
        bg = Image.open(bg_path).convert('RGB')
        W, H = bg.size
        w, h = image_size
        if W >= w and H >= h:
            left = random.randint(0, W - w)
            top = random.randint(0, H - h)
            return bg.crop((left, top, left + w, top + h))
    return None

def project_points_mpl(ax, points_3d):
    """Проецирует 3D-точки на 2D-плоскость с использованием matplotlib."""
    projected = []
    proj = ax.get_proj()
    for x, y, z in points_3d:
        x2d, y2d, _ = proj3d.proj_transform(x, y, z, proj)
        display = ax.transData.transform((x2d, y2d))
        inverted_y = image_size[1] - display[1]
        projected.append([display[0], inverted_y])
    return np.array(projected)

def point_to_segment_distance(p, a, b):
    """Вычисляет расстояние от точки p до отрезка ab."""
    ap = p - a
    ab = b - a
    t = np.clip(np.dot(ap, ab) / np.dot(ab, ab), 0, 1)
    closest = a + t * ab
    return np.linalg.norm(p - closest)

def draw_line_with_texture(ax, p1, p2, color, segments_per_edge, texture_jitter, line_width):
    """Отрисовывает линию с текстурированием (случайными отклонениями точек)."""
    points = np.linspace(p1, p2, segments_per_edge)
    jitter = np.random.normal(scale=texture_jitter, size=points.shape)
    noisy_points = points + jitter
    ax.plot(noisy_points[:, 0], noisy_points[:, 1], noisy_points[:, 2],
            color=color, linewidth=line_width)

def draw_2d_edges(ax, projected_points, edges):
    """Отрисовывает 2D-ребра между проецированными точками."""
    for start, end in edges:
        x1, y1 = projected_points[start]
        x2, y2 = projected_points[end]
        ax.plot([x1, x2], [y1, y2], linewidth=1, color='black')

def get_ordered_vertices(vertices):
    """
    Упорядочивает вершины куба, используя только координаты X и Y.
    Определяет верхнюю грань, упорядочивает её, затем находит нижнюю.
    """
    candidate_faces = [
        [0, 1, 2, 3],
        [4, 5, 6, 7],
        [0, 1, 5, 4],
        [1, 2, 6, 5],
        [2, 3, 7, 6],
        [3, 0, 4, 7]
    ]
    best_face = None
    best_sum_y = -np.inf
    for face in candidate_faces:
        y_sum = sum(vertices[i][1] for i in face)
        if y_sum > best_sum_y:
            best_sum_y = y_sum
            best_face = face
    top_face = best_face

    top_vertices = [(i, vertices[i]) for i in top_face]
    def left_then_lower(item):
        x, y = item[1][0], item[1][1]
        return (x, -y)
    top_vertices_sorted = sorted(top_vertices, key=left_then_lower)
    v0 = top_vertices_sorted[0]

    center = np.mean([v[1][:2] for v in top_vertices], axis=0)
    def angle_from_center(v):
        vec = v[1][:2] - center
        return np.arctan2(vec[1], vec[0])
    angles = [(v[0], angle_from_center(v)) for v in top_vertices]
    v0_angle = dict(angles)[v0[0]]
    rotated_angles = [(i, (a - v0_angle + 2 * np.pi) % (2 * np.pi)) for i, a in angles]
    sorted_top = [i for i, _ in sorted(rotated_angles, key=lambda x: x[1])]

    vertical_edges = []
    for a, b in edges:
        ya, yb = vertices[a][1], vertices[b][1]
        if abs(ya - yb) > 1e-5:
            vertical_edges.append((a, b))
    bottom_sorted = []
    for top_idx in sorted_top:
        match = None
        for a, b in vertical_edges:
            if a == top_idx and b not in sorted_top:
                match = b
                break
            elif b == top_idx and a not in sorted_top:
                match = a
                break
        if match is None:
            raise ValueError(f"Не найдено вертикальное ребро для вершины {top_idx}")
        bottom_sorted.append(match)
    return sorted_top + bottom_sorted

edges = [
    (0, 1), (1, 2), (2, 3), (3, 0),   # Верхняя грань (номера 0-3)
    (4, 5), (5, 6), (6, 7), (7, 4),   # Нижняя грань (номера 4-7)
    (0, 4), (1, 5), (2, 6), (3, 7)    # Вертикальные рёбра
]

def add_noise_to_image(img, noise_density, noise_color_mode='bw'):
    """
    Добавляет случайные шумовые элементы (линии, круги, точки) на изображение.
    Эти элементы могут перекрывать куб, имитируя естественные помехи.
    Возвращает зашумлённое изображение и список описаний шумовых фигур.
    
    Параметры:
    - img: изображение в формате numpy array (BGR)
    - noise_density: интенсивность/количество шумовых фигур
    - noise_color_mode: 'bw' для оттенков серого или 'color' для случайных тёмных цветов
    """
    h, w = img.shape[:2]
    noise_shapes = []
    base_count = int(noise_density * 0.0001 * w * h)
    if base_count < 1:
        base_count = 1
    num_noise_shapes = np.random.randint(1, base_count + 1)
    
    for _ in range(num_noise_shapes):
        shape_type = random.choice(['line', 'circle', 'dot'])
        if noise_color_mode == 'bw':
            intensity = random.randint(0, 128)
            color = (intensity, intensity, intensity)
        else:
            color = (random.randint(0, 128), random.randint(0, 128), random.randint(0, 128))
        
        if shape_type == 'line':
            x1, y1 = random.randint(0, w - 1), random.randint(0, h - 1)
            x2, y2 = random.randint(0, w - 1), random.randint(0, h - 1)
            thickness = random.randint(1, 3)
            cv2.line(img, (x1, y1), (x2, y2), color, thickness)
            noise_shapes.append({'type': 'line', 'start': (x1, y1), 'end': (x2, y2), 'thickness': thickness})
        elif shape_type == 'circle':
            cx, cy = random.randint(0, w - 1), random.randint(0, h - 1)
            radius = random.randint(5, 15)
            if random.random() < 0.5:
                cv2.circle(img, (cx, cy), radius, color, -1)
                filled = True
                thickness_circle = -1
            else:
                thickness_circle = random.randint(1, 3)
                cv2.circle(img, (cx, cy), radius, color, thickness_circle)
                filled = False
            noise_shapes.append({'type': 'circle', 'center': (cx, cy), 'radius': radius,
                                 'filled': filled, 'thickness': thickness_circle})
        elif shape_type == 'dot':
            cx, cy = random.randint(0, w - 1), random.randint(0, h - 1)
            r = random.randint(1, 3)
            cv2.circle(img, (cx, cy), r, color, -1)
            noise_shapes.append({'type': 'circle', 'center': (cx, cy), 'radius': r,
                                 'filled': True, 'thickness': -1})
    
    if random.random() < 0.3:
        if noise_color_mode == 'bw':
            intensity = random.randint(0, 128)
            color = (intensity, intensity, intensity)
        else:
            color = (random.randint(0, 128), random.randint(0, 128), random.randint(0, 128))
        if random.random() < 0.5:
            x1, y1 = random.randint(0, w - 1), random.randint(0, h - 1)
            x2, y2 = random.randint(0, w - 1), random.randint(0, h - 1)
            thickness = random.randint(1, 3)
            cv2.line(img, (x1, y1), (x2, y2), color, thickness)
            noise_shapes.append({'type': 'line', 'start': (x1, y1), 'end': (x2, y2), 'thickness': thickness})
        else:
            cx, cy = random.randint(0, w - 1), random.randint(0, h - 1)
            radius = random.randint(3, 6)
            cv2.circle(img, (cx, cy), radius, color, -1)
            noise_shapes.append({'type': 'circle', 'center': (cx, cy), 'radius': radius,
                                 'filled': True, 'thickness': -1})
    return img, noise_shapes


coco_annotations = {
    "images": [],
    "annotations": [],
    "categories": [
        {
            "supercategory": "shape",
            "id": 1,
            "name": "cube",
            "keypoints": [f"v{i}" for i in range(8)],
            "skeleton": edges
        }
    ]
}

annotation_id = start_index + 1
annotation_path = f"{output_dir}/annotations/coco_keypoints.json"
max_image_id = start_index - 1
max_annotation_id = start_index

if os.path.exists(annotation_path):
    with open(annotation_path, "r") as f:
        existing_data = json.load(f)
    
    existing_images = existing_data.get("images", [])
    existing_annotations = existing_data.get("annotations", [])
    
    if existing_images:
        max_image_id = max(img["id"] for img in existing_images)
    if existing_annotations:
        max_annotation_id = max(ann["id"] for ann in existing_annotations)
    
    start_index = max(start_index, max_image_id + 1)
    annotation_id = max(max_annotation_id + 1, start_index + 1)
    coco_annotations = existing_data
else:
    coco_annotations = {
        "images": [],
        "annotations": [],
        "categories": coco_annotations["categories"]
    }
    annotation_id = start_index + 1


for i in range(start_index, start_index + dataset_size):
    texture_jitter = np.random.uniform(0.0, 0.01)       
    base_line_width = np.random.uniform(0.8, 1.2)
    segments_per_edge = int(np.random.uniform(*segments_per_edge_range))
    image_noise_density = np.random.uniform(0.0, 0.1)

    image_filename = f'cube_{i}.png'
    image_path = f'{output_dir}/images/cubes/{image_filename}'
    temp_path = f'{output_dir}/images/temp_{image_filename}'
    
    fig = plt.figure(figsize=(image_size[0] / 100, image_size[1] / 100), dpi=100)
    ax = fig.add_subplot(111, projection='3d')
    ax.set_facecolor('none')
    fig.patch.set_alpha(0.0)
    plt.axis('off')
    
    vertices = create_noisy_cube()
    rotation = random_rotation_matrix()
    rotated = vertices @ rotation.T
    shift_xy = np.random.uniform(-0.5, 0.5, size=(1, 3))
    shift_xy[0, 2] = 0
    shifted = rotated + shift_xy
    
    dark_color = np.random.uniform(0.0, 0.2, 3)
    for edge in edges:
        p1, p2 = shifted[edge[0]], shifted[edge[1]]
        draw_line_with_texture(ax, p1, p2, dark_color, segments_per_edge, texture_jitter, base_line_width)
    
    ax.set_proj_type('ortho')
    ax.set_xlim([-2, 2])
    ax.set_ylim([-2, 2])
    ax.set_zlim([-2, 2])
    
    fig.canvas.draw()
    projected = project_points_mpl(ax, shifted)
    projected = (projected - (128, 128)) * 256 / 197 + (125, 125)
    plt.savefig(temp_path, bbox_inches='tight', pad_inches=0, transparent=True)
    plt.close(fig)
    
    new_order = get_ordered_vertices(projected)
    projected = projected[new_order]
    
    bg = get_random_background_crop()
    if bg:
        cube_img = Image.open(temp_path).convert('RGBA').resize(image_size)
        bg.paste(cube_img, (0, 0), cube_img)
        bg.save(image_path)
        os.remove(temp_path)
    else:
        os.rename(temp_path, image_path)
    
    noisy_img = cv2.imread(image_path, cv2.IMREAD_COLOR)
    if image_noise_density > 0.09:
        noisy_img, noise_shapes = add_noise_to_image(noisy_img, image_noise_density, noise_color_mode)
    cv2.imwrite(image_path, noisy_img)
    
    keypoints = []
    visibilities = []
    for idx, (x, y) in enumerate(projected):
        point = np.array([x, y])
        visibility = 2
        for e in edges:
            if idx not in e:
                a = projected[e[0]]
                b = projected[e[1]]
                if point_to_segment_distance(point, a, b) < visibility_threshold:
                    visibility = 1
                    break
        if visibility == 2:
            for shape in noise_shapes:
                if shape["type"] == "line":
                    a = np.array(shape["start"])
                    b = np.array(shape["end"])
                    dist = point_to_segment_distance(point, a, b)
                    if dist < shape["thickness"] / 2.0:
                        visibility = 1
                        break
                elif shape["type"] == "circle":
                    center = np.array(shape["center"])
                    dist = np.linalg.norm(point - center)
                    if dist <= shape["radius"]:
                        visibility = 1
                        break
        keypoints.extend([float(x), float(y), visibility])
        visibilities.append(visibility)
    
    # Отрисовка 2D-ребер и ключевых точек для проверки
    cube_img_check = Image.open(image_path).convert('RGBA')
    fig2, ax2 = plt.subplots(figsize=(image_size[0] / 100, image_size[1] / 100), dpi=100)
    ax2.imshow(cube_img_check)
    ax2.set_xlim(0, image_size[0])
    ax2.set_ylim(image_size[1], 0)
    ax2.axis('off')
    draw_2d_edges(ax2, projected, edges)
    for k, (x, y) in enumerate(projected):
        pt_color = 'green' if visibilities[k] == 2 else 'red'
        ax2.scatter(x, y, s=20, c=pt_color, marker='o')
        ax2.text(x, y, str(k), color='black', fontsize=12, weight='bold')
    points_image_path = f'{output_dir}/images/points/cube_{i}_points.png'
    fig2.savefig(points_image_path, bbox_inches='tight', pad_inches=0)
    plt.close(fig2)
    
    coco_annotations["images"].append({
        "file_name": image_filename,
        "id": i,
        "width": image_size[0],
        "height": image_size[1]
    })
    coco_annotations["annotations"].append({
        "id": annotation_id,
        "image_id": i,
        "category_id": 1,
        "keypoints": keypoints,
        "num_keypoints": 8,
        "bbox": [0, 0, image_size[0], image_size[1]],
        "iscrowd": 0,
        "area": image_size[0] * image_size[1]
    })
    annotation_id += 1
    
    if annotation_id % 10 == 0:
        print(f"Аннотация {annotation_id}")

with open(annotation_path, "w") as f:
    json.dump(coco_annotations, f, indent=2)

print(f"Готово! Сгенерировано {dataset_size} изображений начиная с индекса {start_index}.")


Аннотация 5010
Аннотация 5020
Аннотация 5030
Аннотация 5040
Аннотация 5050
Аннотация 5060
Аннотация 5070
Аннотация 5080
Аннотация 5090
Аннотация 5100
Аннотация 5110
Аннотация 5120
Аннотация 5130
Аннотация 5140
Аннотация 5150
Аннотация 5160
Аннотация 5170
Аннотация 5180
Аннотация 5190
Аннотация 5200
Аннотация 5210
Аннотация 5220
Аннотация 5230
Аннотация 5240
Аннотация 5250
Аннотация 5260
Аннотация 5270
Аннотация 5280
Аннотация 5290
Аннотация 5300
Аннотация 5310
Аннотация 5320
Аннотация 5330
Аннотация 5340
Аннотация 5350
Аннотация 5360
Аннотация 5370
Аннотация 5380
Аннотация 5390
Аннотация 5400
Аннотация 5410
Аннотация 5420
Аннотация 5430
Аннотация 5440
Аннотация 5450
Аннотация 5460
Аннотация 5470
Аннотация 5480
Аннотация 5490
Аннотация 5500
Аннотация 5510
Аннотация 5520
Аннотация 5530
Аннотация 5540
Аннотация 5550
Аннотация 5560
Аннотация 5570
Аннотация 5580
Аннотация 5590
Аннотация 5600
Аннотация 5610
Аннотация 5620
Аннотация 5630
Аннотация 5640
Аннотация 5650
Аннотация 5660
Аннотация 

In [8]:
import os
import json
import math
import torch
import numpy as np
from PIL import Image
import random

import torch.utils.data
from torch.utils.data import DataLoader, random_split, ConcatDataset

import torchvision
from torchvision import transforms
from torchvision.models.detection import keypointrcnn_resnet50_fpn
from torchvision.models.detection.keypoint_rcnn import KeypointRCNNPredictor
from torchvision.models.detection import KeypointRCNN_ResNet50_FPN_Weights
from torchvision.transforms import functional as F
from tqdm import tqdm
import matplotlib.pyplot as plt

# ========= Упрощённые аугментации ==========

class CustomTrainTransform:
    def __init__(self, image_size):
        self.image_size = image_size
        
        self.color_jitter = transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.02)

        self.scale_range = (0.95, 1.05)
        self.translate_factor = 0.05

    def __call__(self, image, target):
        w, h = image.size
        center = (w * 0.5, h * 0.5)
        
        scale = random.uniform(*self.scale_range)
        max_dx = self.translate_factor * w
        max_dy = self.translate_factor * h
        translate = (random.uniform(-max_dx, max_dx), random.uniform(-max_dy, max_dy))
        angle = 0 
        shear = 0
        
        image = F.affine(image, angle=angle, translate=translate, scale=scale, shear=shear, center=center)
        image = self.color_jitter(image)
        
        def transform_point(point):
            x, y, vis = point
            new_x = center[0] + scale * (x - center[0]) + translate[0]
            new_y = center[1] + scale * (y - center[1]) + translate[1]
            return [new_x, new_y, vis]
        
        def check_visibility(point, width, height):
            x, y, vis = point
            if x < 0 or x >= width or y < 0 or y >= height:
                vis = 0
            return [x, y, vis]

        if "keypoints" in target:
            new_keypoints = []
            for kp in target["keypoints"]:
                new_kp = [transform_point(p) for p in kp]
                new_kp = [check_visibility(p, w, h) for p in new_kp]
                new_keypoints.append(new_kp)
            target["keypoints"] = torch.as_tensor(new_keypoints, dtype=torch.float32)
        
        if "boxes" in target:
            new_boxes = []
            for box in target["boxes"]:
                xmin, ymin, xmax, ymax = box.tolist()
                pts = np.array([
                    [xmin, ymin],
                    [xmax, ymin],
                    [xmax, ymax],
                    [xmin, ymax]
                ])
                pts_transformed = center + scale * (pts - center) + np.array(translate)
                new_xmin, new_ymin = pts_transformed.min(axis=0)
                new_xmax, new_ymax = pts_transformed.max(axis=0)
                new_boxes.append([new_xmin, new_ymin, new_xmax, new_ymax])
            target["boxes"] = torch.as_tensor(new_boxes, dtype=torch.float32)
        
        image = F.to_tensor(image)
        
        return image, target

class CubeKeypointsDataset(torch.utils.data.Dataset):
    def __init__(self, root, annotation_file, transform=None):
        self.root = root
        self.transform = transform
        
        with open(annotation_file, 'r') as f:
            coco_json = json.load(f)
        
        self.images_info = coco_json["images"]
        annotations_dict = {}
        for ann in coco_json["annotations"]:
            img_id = ann["image_id"]
            if img_id not in annotations_dict:
                annotations_dict[img_id] = []
            annotations_dict[img_id].append(ann)
        self.annotations_dict = annotations_dict
        
    def __len__(self):
        return len(self.images_info)
    
    def __getitem__(self, idx):
        img_info = self.images_info[idx]
        img_id = img_info["id"]
        img_name = img_info["file_name"]
        img_path = os.path.join(self.root, img_name)
        image = Image.open(img_path).convert("RGB")
        
        anns = self.annotations_dict.get(img_id, [])
        boxes, labels, keypoints, areas, iscrowd = [], [], [], [], []
        
        for ann in anns:
            box = ann["bbox"]
            x_min, y_min, w_box, h_box = box
            boxes.append([x_min, y_min, x_min + w_box, y_min + h_box])
            labels.append(1)
            kp_array = np.array(ann["keypoints"]).reshape(-1, 3)
            keypoints.append(kp_array.tolist())
            areas.append(ann["area"])
            iscrowd.append(ann["iscrowd"])
        
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        keypoints = torch.as_tensor(keypoints, dtype=torch.float32)
        areas = torch.as_tensor(areas, dtype=torch.float32)
        iscrowd = torch.as_tensor(iscrowd, dtype=torch.int64)
        
        target = {
            "boxes": boxes,
            "labels": labels,
            "keypoints": keypoints,
            "area": areas,
            "iscrowd": iscrowd,
            "image_id": torch.tensor([img_id])
        }
        
        if self.transform is not None:
            image, target = self.transform(image, target)
        else:
            image = F.to_tensor(image)
        
        return image, target

clean_images_dir = r"cube_dataset/images/cubes_clear"
clean_annotation_path = r"cube_dataset/annotations/coco_keypoints_clear.json"

noisy_images_dir = r"cube_dataset/images/cubes"
noisy_annotation_path = r"cube_dataset/annotations/coco_keypoints.json"

train_transform = CustomTrainTransform(image_size=(256, 256))
val_transform = lambda img, target: (F.to_tensor(img), target)

clean_dataset = CubeKeypointsDataset(root=clean_images_dir, annotation_file=clean_annotation_path, transform=train_transform)
noisy_dataset = CubeKeypointsDataset(root=noisy_images_dir, annotation_file=noisy_annotation_path, transform=train_transform)
combined_dataset = ConcatDataset([clean_dataset, noisy_dataset])

n = len(combined_dataset)
n_train = int(0.6 * n)
n_val = int(0.2 * n)
n_test = n - n_train - n_val

train_dataset, val_dataset, test_dataset = random_split(combined_dataset, [n_train, n_val, n_test])

for ds in [val_dataset, test_dataset]:
    ds.dataset.transform = val_transform
    
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    collate_fn=lambda x: tuple(zip(*x))
)
val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    collate_fn=lambda x: tuple(zip(*x))
)
test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    collate_fn=lambda x: tuple(zip(*x))
)

def get_keypoint_model(num_keypoints=8):
    weights = KeypointRCNN_ResNet50_FPN_Weights.DEFAULT
    model = keypointrcnn_resnet50_fpn(weights=weights)
    in_features = model.roi_heads.keypoint_predictor.kps_score_lowres.in_channels
    model.roi_heads.keypoint_predictor = KeypointRCNNPredictor(in_features, num_keypoints)
    
    in_features_box = model.roi_heads.box_predictor.cls_score.in_features
    num_classes = 2  # фон + куб
    model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(in_features_box, num_classes)
    
    return model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = get_keypoint_model(num_keypoints=8)

pretrained_path = r"finetuned_cube_model_2.pth"
if os.path.exists(pretrained_path):
    print("Загружаем предобученные веса из:", pretrained_path)
    model.load_state_dict(torch.load(pretrained_path))
else:
    print("Предобученные веса не найдены. Обучение с нуля.")

for param in model.backbone.parameters():
    param.requires_grad = False

model.to(device)

fine_tuning_lr = 1e-7  
num_fine_tune_epochs = 8 

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                              lr=fine_tuning_lr, weight_decay=0.0005)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=1e-4,
    steps_per_epoch=len(train_loader),
    epochs=num_fine_tune_epochs
)

scaler = torch.amp.GradScaler("cuda")

def train_one_epoch(model, optimizer, scheduler, data_loader, device, scaler):
    model.train()
    epoch_loss = 0.0
    progress_bar = tqdm(data_loader, desc="Training", leave=False)
    
    for images, targets in progress_bar:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
        scaler.scale(losses).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        loss_val = losses.item()
        epoch_loss += loss_val
        progress_bar.set_postfix(loss=loss_val)
    
    return epoch_loss / len(data_loader)

@torch.no_grad()
def validate(model, data_loader, device):
    was_training = model.training
    model.train()  
    val_loss = 0.0
    progress_bar = tqdm(data_loader, desc="Validating", leave=False)
    
    for images, targets in progress_bar:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        val_loss += losses.item()
        progress_bar.set_postfix(loss=losses.item())
    
    if was_training:
        model.train()
    
    return val_loss / len(data_loader)


Загружаем предобученные веса из: finetuned_cube_model_2.pth


C:\Users\andre\AppData\Local\Temp\ipykernel_4228\921194213.py:240: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(pretrained_path))


In [9]:
unfreeze_epoch = 0 

print("Разморозка слоёв backbone.body.layer3 и layer4...")
for name, parameter in model.backbone.body.named_parameters():
    if name.startswith("layer3") or name.startswith("layer4"):
        parameter.requires_grad = True
        
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                              lr=fine_tuning_lr, weight_decay=0.0005)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=1e-4,
    steps_per_epoch=len(train_loader),
    epochs=num_fine_tune_epochs
    )

for epoch in range(num_fine_tune_epochs):
    print(f"Эпоха {epoch+1}/{num_fine_tune_epochs}")
    train_loss = train_one_epoch(model, optimizer, scheduler, train_loader, device, scaler)
    val_loss = validate(model, val_loader, device)
    print(f"Train Loss: {train_loss:.4f}  Val Loss: {val_loss:.4f}")
    
    # if epoch + 1 == unfreeze_epoch:
    #     print("Разморозка слоёв backbone.body.layer3 и layer4...")
    #     for name, parameter in model.backbone.body.named_parameters():
    #         if name.startswith("layer3") or name.startswith("layer4"):
    #             parameter.requires_grad = True
    #     # Пересоздаем оптимизатор и scheduler с обновлёнными параметрами
    #     optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
    #                                   lr=fine_tuning_lr, weight_decay=0.0005)
    #     scheduler = torch.optim.lr_scheduler.OneCycleLR(
    #         optimizer,
    #         max_lr=1e-4,
    #         steps_per_epoch=len(train_loader),
    #         epochs=num_fine_tune_epochs - epoch
    #     )

    if epoch+2 % 3 == 0:
        # Можно сохранять модель после каждой эпохи, если нужно
        torch.save(model.state_dict(), f"finetuned_cube_model_3_epoch_{epoch+1}.pth")

torch.save(model.state_dict(), "finetuned_cube_model_3.pth")
print("Финальная модель сохранена в finetuned_cube_model_3.pth")


Разморозка слоёв backbone.body.layer3 и layer4...
Эпоха 1/8


Training:   0%|          | 0/3000 [00:00<?, ?it/s]C:\Users\andre\AppData\Local\Temp\ipykernel_4228\921194213.py:279: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Train Loss: 0.4583  Val Loss: 0.5140
Эпоха 2/8


Train Loss: 0.4673  Val Loss: 0.5278
Эпоха 3/8


KeyboardInterrupt: 

In [10]:
import cv2
import numpy as np
import os
import torch
import matplotlib.pyplot as plt
import csv

torch.save(model.state_dict(), "finetuned_cube_model_3.pth")
print("Финальная модель сохранена в finetuned_cube_model_3.pth")

output_dir = 'cube_dataset/images/predicts'
os.makedirs(output_dir, exist_ok=True) 

def tensor_to_numpy(img_tensor):
    img_np = img_tensor.cpu().permute(1, 2, 0).numpy()
    return img_np

def visualize_prediction(i, image_np, gt_keypoints, pred_keypoints, skeleton, mse, score_thresh=2.0):
    """
    image_np: изображение в формате numpy (RGB)
    gt_keypoints: numpy-массив истинных ключевых точек [K, 3] (x, y, visibility)
    pred_keypoints: numpy-массив предсказанных ключевых точек [K, 3] (x, y, score)
    skeleton: список пар индексов, определяющих рёбра (например, [[0, 1], [1, 2], ...])
    mse: значение mse по ключевым точкам
    score_thresh: порог уверенности для отображения предсказанных точек (не используется в данном примере)
    """
    plt.figure(figsize=(8, 8))
    plt.imshow(image_np)

    for j, (x, y, vis) in enumerate(gt_keypoints):
        plt.scatter(x, y, s=50, c='lime', marker='o', label='Ground Truth' if j == 0 else "")
        plt.text(x, y, str(j), color='black', fontsize=12, weight='bold')
    
    for j, (x, y, score) in enumerate(pred_keypoints):
        plt.scatter(x, y, s=50, c='red', marker='x', label='Prediction' if j == 0 else "")
        plt.text(x, y, str(j), color='yellow', fontsize=12, weight='bold')
    
    for start_idx, end_idx in skeleton:
        pt1 = gt_keypoints[start_idx]
        pt2 = gt_keypoints[end_idx]
        plt.plot([pt1[0], pt2[0]], [pt1[1], pt2[1]], c='lime', linestyle='--', linewidth=2)
    
    for start_idx, end_idx in skeleton:
        pt1 = pred_keypoints[start_idx]
        pt2 = pred_keypoints[end_idx]
        plt.plot([pt1[0], pt2[0]], [pt1[1], pt2[1]], c='red', linestyle='-', linewidth=2)
    
    gt_points = [f"({int(x)}, {int(y)})" for x, y, _ in gt_keypoints]
    pred_points = [f"({int(x)}, {int(y)})" for x, y, _ in pred_keypoints]
    annotation = f"True: {gt_points}\nPred: {pred_points}\nMSE: {mse:.2f}"
    plt.gcf().text(0.02, 0.02, annotation, fontsize=10, bbox=dict(facecolor='white', alpha=0.5))
    
    plt.axis('off')
    plt.legend()
    
    points_image_path = f'{output_dir}/finetuned_cube_model_cube_{i}_points.png'
    plt.savefig(points_image_path, bbox_inches='tight', pad_inches=0, transparent=True)
    plt.close()

@torch.no_grad()
def test_and_visualize(model, data_loader, device, score_thresh=2.0, visualize_every=1):
    model.eval()
    total_mse = 0.0
    count = 0

    mse_results = []
    skeleton = [
        [0, 1],
        [1, 2],
        [2, 3],
        [3, 0],
        [4, 5],
        [5, 6],
        [6, 7],
        [7, 4],
        [0, 4],
        [1, 5],
        [2, 6],
        [3, 7]
    ]

    for batch_idx, (images, targets) in enumerate(data_loader):
        images = [img.to(device) for img in images]
        outputs = model(images)  # Предсказания без targets
        
        for i, (img_tensor, output, target) in enumerate(zip(images, outputs, targets)):
            if len(output["keypoints"]) == 0 or target["keypoints"].shape[0] == 0:
                continue
            pred_kp = output["keypoints"][0].cpu().numpy()  # [K, 3]
            true_kp = target["keypoints"][0].cpu().numpy()    # [K, 3]
            
            mse = ((pred_kp[:, :2] - true_kp[:, :2]) ** 2).sum(axis=1).mean()
            total_mse += mse
            count += 1
            
            # Если mse меньше 5, считаем аннотацию корректной
            correct = mse < 5.0
            mse_results.append([count, mse, 'correct' if correct else 'incorrect'])
            
            if count % visualize_every == 0:
                image_np = tensor_to_numpy(img_tensor)
                visualize_prediction(7000 + count, image_np, true_kp, pred_kp, skeleton, mse, score_thresh)
    
    mse_csv_path = os.path.join('cube_dataset', 'mse_results_3.csv')
    os.makedirs(os.path.dirname(mse_csv_path), exist_ok=True)
    with open(mse_csv_path, mode='w', newline='') as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(["image_index", "mse", "annotation"])
        for row in mse_results:
            writer.writerow(row)
    
    # Вычисляем процент корректных изображений и среднее mse по корректным
    correct_mses = [row[1] for row in mse_results if row[2] == 'correct']
    percent_correct = (len(correct_mses) / len(mse_results)) * 100 if mse_results else 0
    mean_correct_mse = np.mean(correct_mses) if correct_mses else 0
    
    print(f"Test MSE (по ключевым точкам): {total_mse/count if count else 0:.4f}")
    print(f"Процент корректных аннотаций: {percent_correct:.2f}%")
    print(f"Среднее MSE для корректных аннотаций: {mean_correct_mse:.4f}")

# Пример тестирования с визуализацией
# Убедитесь, что model, test_loader и device корректно определены
test_and_visualize(model, test_loader, device)


Финальная модель сохранена в finetuned_cube_model_3.pth
Test MSE (по ключевым точкам): 55.5167
Процент корректных аннотаций: 93.45%
Среднее MSE для корректных аннотаций: 0.8299
